In [1]:
import pickle

In [2]:
with open("climate.pkl", "rb") as f:
    data = pickle.load(f)

In [3]:
len(data)

136070

In [13]:
from __future__ import annotations
from geopy.geocoders import Nominatim
from dataclasses import dataclass
import xarray as xr
import pandas as pd
import numpy as np

@dataclass
class Location:
    name: str
    latitude: float
    longitude: float

_geolocator = None

def geocode_city(city: str, user_agent: str = "pogoda-app") -> Location:
    """Geocode a city name into coordinates using Nominatim.

    Parameters
    ----------
    city: str
        City name (can include country, e.g., "Szczecin, Poland").
    user_agent: str
        Required by Nominatim usage policy.

    Returns
    -------
    Location

    Raises
    ------
    ValueError: if no result is found.
    """
    global _geolocator
    if _geolocator is None:
        _geolocator = Nominatim(user_agent=user_agent, timeout=10)
    result = _geolocator.geocode(city)
    if not result:
        raise ValueError(f"City not found: {city}")
    return Location(name=result.address, latitude=result.latitude, longitude=result.longitude)


In [6]:
print(city)

Location(name='Trieste, Friuli-Venezia Giulia, 34121-34151, Italia', latitude=45.6496485, longitude=13.7772781)


In [7]:
from sklearn.neighbors import BallTree

In [19]:
locations =  np.array(list(data.keys()))
radians = np.radians(locations)
tree = BallTree(radians, metric = "haversine")

Closest place: [53.35 -6.25] index: 82263
Distance: 0.68 km


In [134]:
city = geocode_city("bijelo pole")
print(city)
lbound_year = 2018
ubound_year = 2023


my_city_deg = (53.3498, -6.2603)  # Dublin
my_city_rad = np.radians([(city.latitude, city.longitude)])

dist_rad, ind = tree.query(my_city_rad, k=1)

idx = ind[0, 0]
dist_km = dist_rad[0, 0] * 6371.0088  # Earth mean radius in km

print("Closest place:", locations[idx], "index:", idx)
print(f"Distance: {dist_km:.2f} km")
place = (float(locations[idx][0]),float(locations[idx][1]))
climate : Dict[int, List[float,float]] = data[place]
temp = [0]*12
percip = [0]*12
years = 0
for year in range(lbound_year, ubound_year+1):
    if year in climate:
        years = years +1
        for i in range(0,12):
            temp[i] = temp[i] + climate[year][i][0]
            percip[i] = percip[i] + climate[year][i][1]
    else:
        print("data for year %i is missing" % year)
avg_temp = [t/years for t in temp]
avg_percip = [p/years for p in percip]

print(classify_koppen(avg_temp, avg_percip,city.latitude))
print(classify_trewartha(avg_temp, avg_percip,city.latitude))

Location(name='Bijelo Polje, Opština Bijelo Polje, Crna Gora / Црна Гора', latitude=43.0341595, longitude=19.7473559)
Closest place: [43.05 19.75] index: 37605
Distance: 1.77 km
('Cfb', {'annual_mean_temp': 10.229319444444444, 'annual_precip': 1196.4166666666667, 'warmest': 19.950833333333335, 'coldest': -0.17833333333333332, 'months_ge_10': 6, 'precip_summer': 483.05, 'precip_winter': 713.3666666666666, 'summer_share': 0.4037473009681688, 'winter_share': 0.5962526990318311, 'dryness_threshold_R': 34.458638888888885, 'group': 'C', 'second': 'f', 'third': 'b'})
('Dfbl', {'annual_mean_temp': 10.229319444444444, 'annual_precip': 1196.4166666666667, 'warmest': 19.950833333333335, 'coldest': -0.17833333333333332, 'months_ge_10': 6, 'precip_summer': 483.05, 'precip_winter': 713.3666666666666, 'summer_share': 0.4037473009681688, 'winter_share': 0.5962526990318311, 'dryness_threshold_R': 34.458638888888885, 'four_letter': True, 'thermal_scale': 'l', 'group': 'D', 'second': 'f', 'third': 'b'})


##### 

In [67]:
from __future__ import annotations
from typing import List, Dict, Tuple

class KoppenClassificationError(ValueError):
    pass

def classify_koppen(temps_c: List[float], precip_mm: List[float], latitude: float) -> Tuple[str, Dict[str, float]]:
    """Classify climate using simplified Köppen system.

    Notes
    -----
    - Uses -3°C isotherm between C and D (common modern variant); adjust if needed.
    - Requires 12 monthly mean temps and 12 monthly precip totals.
    - Simplified tertiary letters for temperate/continental groups.
    """
    if len(temps_c) != 12 or len(precip_mm) != 12:
        raise KoppenClassificationError("Need 12 monthly temperature and precipitation values")

    # Basic derived metrics
    annual_mean_temp = sum(temps_c) / 12.0
    annual_precip = sum(precip_mm)
    warmest = max(temps_c)
    coldest = min(temps_c)
    months_ge_10 = sum(1 for t in temps_c if t >= 10.0)

    # Hemisphere aware: define summer/winter half-year
    # Northern: Apr-Sep as summer (months 4-9); Southern: Oct-Mar (months 10-3)
    if latitude >= 0:
        summer_indices = [3,4,5,6,7,8]  # 0-based for Apr-Sep
        winter_indices = [9,10,11,0,1,2]
    else:
        summer_indices = [9,10,11,0,1,2]
        winter_indices = [3,4,5,6,7,8]
    precip_summer = sum(precip_mm[i] for i in summer_indices)
    precip_winter = sum(precip_mm[i] for i in winter_indices)

    summer_share = precip_summer / annual_precip if annual_precip > 0 else 0
    winter_share = precip_winter / annual_precip if annual_precip > 0 else 0

    # Dryness threshold (Köppen B) R
    if summer_share >= 0.70:
        R = 2 * annual_mean_temp + 28
    elif winter_share >= 0.70:
        R = 2 * annual_mean_temp
    else:
        R = 2 * annual_mean_temp + 14

    details = dict(
        annual_mean_temp=annual_mean_temp,
        annual_precip=annual_precip,
        warmest=warmest,
        coldest=coldest,
        months_ge_10=months_ge_10,
        precip_summer=precip_summer,
        precip_winter=precip_winter,
        summer_share=summer_share,
        winter_share=winter_share,
        dryness_threshold_R=R,
    )

    # Main group determination (order matters: arid first)
    if annual_precip < R:
        main = 'B'
    elif min(temps_c) >= 18.0:
        main = 'A'
    elif warmest < 10.0:
        main = 'E'
    elif coldest > -3.0:
        main = 'C'
    else:
        main = 'D'

    # Special polar subtypes
    if main == 'E':
        subtype = 'T' if warmest > 0.0 else 'F'
        code = main + subtype
        details['group'] = main
        details['subtype'] = subtype
        return code, details

    # Arid subdivisions (B)
    if main == 'B':
        if annual_precip < 0.5 * R:
            sub1 = 'W'  # desert
        else:
            sub1 = 'S'  # steppe
        # Temperature qualifier
        if annual_mean_temp >= 18.0:
            sub2 = 'h'
        else:
            sub2 = 'k'
        code = main + sub1 + sub2
        details['group'] = main
        details['sub1'] = sub1
        details['sub2'] = sub2
        return code, details

    # Seasonality second letter for A, C, D
    # Monthly ratio criteria (classical):
    # 's': driest summer month < 40 mm AND < 1/3 of wettest winter month
    # 'w': driest winter month < (wettest summer month / 10)
    # else 'f'
    summer_months = [precip_mm[i] for i in summer_indices]
    winter_months = [precip_mm[i] for i in winter_indices]
    driest_summer = min(summer_months) if summer_months else 0
    driest_winter = min(winter_months) if winter_months else 0
    wettest_winter = max(winter_months) if winter_months else 0
    wettest_summer = max(summer_months) if summer_months else 0
    #print(f" driest_summer {driest_summer} wettest_winter {wettest_winter}")
    if driest_summer < 40 and wettest_winter and driest_summer < (wettest_winter/3.0):
        second = 's'
    elif wettest_summer and driest_winter < (wettest_summer/10.0):
        second = 'w'
    else:
        second = 'f'

    # Temperature third letter for C/D
    if main in ('C','D'):
        if warmest >= 22.0 and months_ge_10 >= 4:
            third = 'a'
        elif months_ge_10 >= 4 and warmest < 22.0:
            third = 'b'
        elif months_ge_10 >= 1:
            third = 'c'
        else:
            third = 'd'  # very cold winters
    else:
        third = ''

    # Tropical sub letters (A)
    if main == 'A':
        # Af: no dry month (all months >= 60 mm) - simplified using 60 mm absolute
        if all(p >= 60 for p in precip_mm):
            second = 'f'
        else:
            # Am vs Aw: If short dry season but not pronounced vs strong winter dryness
            # Simplified: choose 'w' if any winter month < 60 and precip_winter < precip_summer
            if any(precip_mm[i] < 60 for i in winter_indices) and precip_winter < precip_summer:
                second = 'w'
            else:
                second = 'm'
        code = main + second
        details['group'] = main
        details['second'] = second
        return code, details

    code = main + second + third
    details['group'] = main
    details['second'] = second
    details['third'] = third
    return code, details


In [68]:
from __future__ import annotations
from typing import List, Dict, Tuple

class TrewarthaClassificationError(ValueError):
    pass

def classify_trewartha(temps_c: List[float], precip_mm: List[float], latitude: float) -> Tuple[str, Dict[str, float]]:
    """Classify climate using a simplified Trewartha system.

     Implementation notes
     --------------------
     1. Dry (B) climates determined first using Köppen-like dryness threshold.
     2. Canonical groups after dryness check:
         - A: All 12 months >= 18°C (true tropical)
         - C: Subtropical (m10 >= 8) (at least 8 months >=10°C but not all >=18°C)
         - D: Temperate/Oceanic (4 <= m10 <= 7)
         - E: Boreal/Subpolar (1 <= m10 <= 3)
         - F: Polar (m10 == 0)
    3. Four-letter code for non-arid (non-B) climates:
        [Group][Precip][Summer][Thermal]
        - Precip: s (dry summer), w (dry winter), f (no pronounced dry season)
        - Summer: a (hot, warmest ≥22°C), b (warm, warmest <22°C but ≥4 months ≥10°C), c (cool/short summer: 1–3 months ≥10°C)
        - Thermal (Universal Thermal Scale, based on ANNUAL mean temperature, Wikipedia ranges):
            i: ≥35°C (severely hot)
            h: 28–34.9°C (very hot)
            a: 22.2–27.9°C (hot)
            b: 18–22.1°C (warm)
            l: 10–17.9°C (mild)
            k: 0.1–9.9°C (cool)
            o: −9.9–0°C (cold)
            c: −24.9––10°C (very cold)
            d: −39.9––25°C (severely cold)
            e: ≤−40°C (excessively cold)
      (Note: Previous implementation used a winter severity letter; this has been replaced for conformity.)
    4. Arid B climates retain 3-letter form (BW/BS + h/k) for familiarity.
    5. Seasonality letters (f, s, w) for A, C, D if not dry: based on ratio comparisons.
    6. B climates subdivided into BW/BS (desert/steppe) and temperature qualifier h/k (>=18°C mean).
    7. For polar (E/F here): still produce four-letter code with thermal scale.
    8. This pragmatic version may be refined later.
    """
    if len(temps_c) != 12 or len(precip_mm) != 12:
        raise TrewarthaClassificationError("Need 12 monthly temperature and precipitation values")

    annual_mean_temp = sum(temps_c)/12.0
    annual_precip = sum(precip_mm)
    warmest = max(temps_c)
    coldest = min(temps_c)
    m10 = sum(1 for t in temps_c if t >= 10.0)

    # Hemisphere seasons (align with Köppen function choices)
    if latitude >= 0:
        summer_indices = [3,4,5,6,7,8]  # Apr-Sep
        winter_indices = [9,10,11,0,1,2]
    else:
        summer_indices = [9,10,11,0,1,2]
        winter_indices = [3,4,5,6,7,8]
    precip_summer = sum(precip_mm[i] for i in summer_indices)
    precip_winter = sum(precip_mm[i] for i in winter_indices)
    summer_share = precip_summer/annual_precip if annual_precip>0 else 0
    winter_share = precip_winter/annual_precip if annual_precip>0 else 0

    # Dryness threshold (same formula used in simplified Köppen implementation)
    if summer_share >= 0.70:
        R = 2*annual_mean_temp + 28
    elif winter_share >= 0.70:
        R = 2*annual_mean_temp
    else:
        R = 2*annual_mean_temp + 14

    details = dict(
        annual_mean_temp=annual_mean_temp,
        annual_precip=annual_precip,
        warmest=warmest,
        coldest=coldest,
        months_ge_10=m10,
        precip_summer=precip_summer,
        precip_winter=precip_winter,
        summer_share=summer_share,
        winter_share=winter_share,
        dryness_threshold_R=R,
    )

    # Dry group overrides others
    if annual_precip < R:
        if annual_precip < 0.5 * R:
            sub1 = 'W'
        else:
            sub1 = 'S'
        temp_qual = 'h' if annual_mean_temp >= 18.0 else 'k'
        code = 'B' + sub1 + temp_qual
        details['group'] = 'B'
        details['sub1'] = sub1
        details['temp_qual'] = temp_qual
        details['four_letter'] = False
        return code, details

    # Non-dry classification by canonical rules
    if all(t >= 18.0 for t in temps_c):
        group = 'A'
    elif m10 >= 8:
        group = 'C'
    elif 4 <= m10 <= 7:
        group = 'D'
    elif 1 <= m10 <= 3:
        group = 'E'
    else:
        group = 'F'

    def thermal_letter(mean_t: float) -> str:
        # Descending thresholds; ranges per Universal Thermal Scale
        if mean_t >= 35.0:
            return 'i'
        if mean_t >= 28.0:
            return 'h'
        if mean_t >= 22.2:
            return 'a'
        if mean_t >= 18.0:
            return 'b'
        if mean_t >= 10.0:
            return 'l'
        if mean_t >= 0.1:
            return 'k'
        if mean_t >= -9.9:  # -9.9 to 0.0
            return 'o'
        if mean_t >= -24.9:
            return 'c'
        if mean_t >= -39.9:
            return 'd'
        return 'e'

    if group == 'F':  # Polar (retain logic but new thermal letter)
        subtype = 'T' if warmest >= 0.0 and warmest < 10.0 else ('F' if warmest < 0.0 else 'T')
        second = 'f'
        summer_letter = 'c'
        t_letter = thermal_letter(annual_mean_temp)
        code = group + second + summer_letter + t_letter
        details['group'] = group
        details['subtype'] = subtype
        details['second'] = second
        details['third'] = summer_letter
        details['thermal_scale'] = t_letter
        details['four_letter'] = True
        return code, details

    # Precipitation seasonality second letter using monthly ratio criteria (Mediterranean / monsoonal detection)
    # 's': driest summer month < 40 mm AND < 1/3 of wettest winter month
    # 'w': driest winter month < (wettest summer month / 10)
    # else 'f'
    summer_months = [precip_mm[i] for i in summer_indices]
    winter_months = [precip_mm[i] for i in winter_indices]
    driest_summer = min(summer_months) if summer_months else 0
    driest_winter = min(winter_months) if winter_months else 0
    wettest_winter = max(winter_months) if winter_months else 0
    wettest_summer = max(summer_months) if summer_months else 0
    if driest_summer < 40 and wettest_winter and driest_summer < (wettest_winter / 3.0):
        second = 's'
    elif wettest_summer and driest_winter < (wettest_summer / 10.0):
        second = 'w'
    else:
        second = 'f'

    # Temperature qualifier: a = warmest >= 22, b = warmest < 22 but at least 4 months >=10, c/d rarely used here; keep simple
    if group in ('C','D','E','A'):
        if warmest >= 22.0:
            third = 'a'
        elif m10 >= 4:
            third = 'b'
        else:
            third = 'c'
    else:
        third = ''

    # Universal thermal scale letter based on annual mean temperature
    t_letter = thermal_letter(annual_mean_temp)

    if third:  # non-arid group
        code = group + second + third + t_letter
        details['four_letter'] = True
        details['thermal_scale'] = t_letter
    else:  # pathological fallback
        code = group + second + t_letter
        details['four_letter'] = False
        details['thermal_scale'] = t_letter
    details['group'] = group
    details['second'] = second
    details['third'] = third
    return code, details


('Cfb',
 {'annual_mean_temp': 10.880595833333333,
  'annual_precip': 722.8349999999998,
  'warmest': 16.5021,
  'coldest': 6.257600000000001,
  'months_ge_10': 6,
  'precip_summer': 343.91999999999996,
  'precip_winter': 378.91499999999996,
  'summer_share': 0.4757932308203118,
  'winter_share': 0.5242067691796884,
  'dryness_threshold_R': 35.76119166666666,
  'group': 'C',
  'second': 'f',
  'third': 'b'})

In [135]:
top_cities = [
  {"city": "London", "country": "United Kingdom"},
  {"city": "Berlin", "country": "Germany"},
  {"city": "Madrid", "country": "Spain"},
  {"city": "Rome", "country": "Italy"},
  {"city": "Paris", "country": "France"},
  {"city": "Vienna", "country": "Austria"},
  {"city": "Amsterdam", "country": "Netherlands"},
  {"city": "Brussels", "country": "Belgium"},
  {"city": "Stockholm", "country": "Sweden"},
  {"city": "Warsaw", "country": "Poland"},
  {"city": "Oslo", "country": "Norway"},
  {"city": "Copenhagen", "country": "Denmark"},
  {"city": "Helsinki", "country": "Finland"},
  {"city": "Lisbon", "country": "Portugal"},
  {"city": "Prague", "country": "Czech Republic"},
  {"city": "Budapest", "country": "Hungary"},
  {"city": "Dublin", "country": "Ireland"},
  {"city": "Athens", "country": "Greece"},
  {"city": "Edinburgh", "country": "United Kingdom"},
  {"city": "Barcelona", "country": "Spain"},
  {"city": "Munich", "country": "Germany"},
  {"city": "Zurich", "country": "Switzerland"},
  {"city": "Hamburg", "country": "Germany"},
  {"city": "Frankfurt", "country": "Germany"},
  {"city": "Warsaw", "country": "Poland"},
  {"city": "Milan", "country": "Italy"},
  {"city": "Belgrade", "country": "Serbia"},
  {"city": "Bucharest", "country": "Romania"},
  {"city": "Bratislava", "country": "Slovakia"},
  {"city": "Ljubljana", "country": "Slovenia"},
  {"city": "Zagreb", "country": "Croatia"},
  {"city": "Vilnius", "country": "Lithuania"},
  {"city": "Riga", "country": "Latvia"},
  {"city": "Tallinn", "country": "Estonia"},
  {"city": "Oslo", "country": "Norway"},
  {"city": "Kiev", "country": "Ukraine"},
  {"city": "Sofia", "country": "Bulgaria"},
  {"city": "Skopje", "country": "North Macedonia"},
  {"city": "Podgorica", "country": "Montenegro"},
  {"city": "Tirana", "country": "Albania"},
  {"city": "Sarajevo", "country": "Bosnia and Herzegovina"},
  {"city": "Chisinau", "country": "Moldova"},
  {"city": "Pristina", "country": "Kosovo"},
  {"city": "Zagreb", "country": "Croatia"},
  {"city": "Baku", "country": "Azerbaijan"},
  {"city": "Tbilisi", "country": "Georgia"},
  {"city": "Yerevan", "country": "Armenia"},
  {"city": "Chisinau", "country": "Moldova"},
  {"city": "Belgrade", "country": "Serbia"},
  {"city": "Lviv", "country": "Ukraine"},
  {"city": "Kharkiv", "country": "Ukraine"},
  {"city": "Gothenburg", "country": "Sweden"},
  {"city": "Bristol", "country": "United Kingdom"},
  {"city": "Manchester", "country": "United Kingdom"},
  {"city": "Leeds", "country": "United Kingdom"},
  {"city": "Birmingham", "country": "United Kingdom"},
  {"city": "Liverpool", "country": "United Kingdom"},
  {"city": "Cardiff", "country": "United Kingdom"},
  {"city": "Glasgow", "country": "United Kingdom"},
  {"city": "Newcastle upon Tyne", "country": "United Kingdom"},
  {"city": "Naples", "country": "Italy"},
  {"city": "Florence", "country": "Italy"},
  {"city": "Venice", "country": "Italy"},
  {"city": "Marseille", "country": "France"},
  {"city": "Lyon", "country": "France"},
  {"city": "Nice", "country": "France"},
  {"city": "Lille", "country": "France"},
  {"city": "Leipzig", "country": "Germany"},
  {"city": "Stuttgart", "country": "Germany"},
  {"city": "Cologne", "country": "Germany"},
  {"city": "Geneva", "country": "Switzerland"},
  {"city": "Basel", "country": "Switzerland"},
  {"city": "Lausanne", "country": "Switzerland"},
  {"city": "Lugano", "country": "Switzerland"},
  {"city": "Lucerne", "country": "Switzerland"},
  {"city": "Chur", "country": "Switzerland"},
  {"city": "Brno", "country": "Czech Republic"},
  {"city": "Pilsen", "country": "Czech Republic"},
  {"city": "Cluj-Napoca", "country": "Romania"},
  {"city": "Iasi", "country": "Romania"},
  {"city": "Timisoara", "country": "Romania"},
  {"city": "Constanta", "country": "Romania"},
  {"city": "Dresden", "country": "Germany"},
  {"city": "Bremen", "country": "Germany"},
  {"city": "Essen", "country": "Germany"},
  {"city": "Hannover", "country": "Germany"},
  {"city": "Leuven", "country": "Belgium"},
  {"city": "Antwerp", "country": "Belgium"},
  {"city": "Ghent", "country": "Belgium"},
  {"city": "Bruges", "country": "Belgium"},
  {"city": "Gent", "country": "Belgium"},
  {"city": "Kraków", "country": "Poland"},
  {"city": "Gdansk", "country": "Poland"},
  {"city": "Wroclaw", "country": "Poland"},
  {"city": "Poznan", "country": "Poland"},
  {"city": "Lviv", "country": "Ukraine"},
  {"city": "Odessa", "country": "Ukraine"}
]

In [140]:
SD = {}
for cdata in top_cities:
    city = geocode_city(cdata["city"])
    print(city)
    my_city_rad = np.radians([(city.latitude, city.longitude)])
    dist_rad, ind = tree.query(my_city_rad, k=1)
    idx = ind[0, 0]
    dist_km = dist_rad[0, 0] * 6371.0088  # Earth mean radius in km
    print("Closest place:", locations[idx], "index:", idx)
    print(f"Distance: {dist_km:.2f} km")
    if (dist_km > 10):
        print("too far")
        continue
    place = (float(locations[idx][0]),float(locations[idx][1]))
    SD[place] = data[place]


Location(name='London, Greater London, England, United Kingdom', latitude=51.4893335, longitude=-0.1440551)
Closest place: [51.45 -0.15] index: 73018
Distance: 4.39 km
Location(name='Berlin, Deutschland', latitude=52.510885, longitude=13.3989367)
Closest place: [52.55 13.35] index: 78530
Distance: 5.47 km
Location(name='Madrid, Comunidad de Madrid, España', latitude=40.4167047, longitude=-3.7035825)
Closest place: [40.45 -3.75] index: 29269
Distance: 5.40 km
Location(name='Roma, Roma Capitale, Lazio, Italia', latitude=41.8933203, longitude=12.4829321)
Closest place: [41.85 12.45] index: 34070
Distance: 5.54 km
Location(name='Paris, Île-de-France, France métropolitaine, France', latitude=48.8534951, longitude=2.3483915)
Closest place: [48.85  2.35] index: 61091
Distance: 0.41 km
Location(name='Wien, Österreich', latitude=48.2083537, longitude=16.3725042)
Closest place: [48.25 16.35] index: 58319
Distance: 4.92 km
Location(name='Amsterdam, Noord-Holland, Nederland', latitude=52.3730796, 

In [147]:
with  open("climate_small.tsv","w") as outF:
    for key in SD.keys():
        outF.write(f"{key[0]}\t{key[1]}\t{SD[key]}\n")

In [148]:
with open("climate_small.pkl", "wb") as f:
    pickle.dump(SD, f)